In [1]:
import os
import thrember
import numpy as np
import pandas as pd
import polars as pl
import altair as alt
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, roc_curve
alt.renderers.enable('default')

/Users/satvik/Desktop/projects/multi-label-static-analysis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RendererRegistry.enable('default')

In [41]:
thrember.download_dataset(download_dir="data/ember_data", file_type='APK')

Unzipping...
Unzipping...


In [55]:
data_dir = "data/ember_data"

In [56]:
train_df, test_df, challenge_df = thrember.read_metadata(data_dir)

In [57]:
# Add a 'week' column to the dataframe
plotdf = pl.concat([train_df, test_df])
start_date = pd.Timestamp("2023-09-24")
plotdf = plotdf.with_columns(
    pl.from_epoch("first_submission_date", time_unit="s").alias("first_submission_dt")
)
plotdf = plotdf.with_columns(
    (
        (pl.col("first_submission_dt") - pl.lit(start_date)).dt.total_days() // 7
    ).cast(pl.Int64).alias("week")
)

print(plotdf.shape)

# Plot file types across weeks
gbdf = plotdf.group_by(["file_type", "week"]).agg(pl.len().alias("count"))
alt.Chart(gbdf).mark_bar().encode(
    alt.X('week:O', axis=alt.Axis(title='Week First Seen')),
    alt.Y('count:Q', axis=alt.Axis(title='File Type')),
        alt.Color('file_type:N', scale=alt.Scale(range=["#4c78a8", "#54a24b", "#f58518",  "#88d27a",  "#9ecae9", "#ffbf79"]),
              legend=alt.Legend(values=["Win32", "Win64", "Dot_Net", "APK", "ELF", "PDF"]))
)

(640000, 16)


alt.Chart(...)

In [58]:
# Get number of occurrences of each family
family_counts = plotdf.select(
    pl.col("family").value_counts().alias("family_counts")
).unnest("family_counts")
family_counts = (
    plotdf.filter(pl.col("family").is_not_null())
          .select(pl.col("family").value_counts())
          .unnest("family")
          .sort("count", descending=True)
)
family_counts.head(10)

family,count
str,u32
"""jiagu""",19028
"""spymax""",14214
"""wapron""",13814
"""mobidash""",7998
"""wacatac""",7340
"""metasploit""",7292
"""boogr""",6558
"""hiddad""",6488
"""adlo""",6212


In [59]:
plotdf_explode = plotdf.filter(pl.col("behavior").list.len() > 0).explode("behavior")
behavior_counts = (plotdf_explode.group_by("behavior").agg(pl.len().alias("count")).sort("count", descending=True))
behavior_counts.head(10)

behavior,count
str,u32
"""phishing""",43556
"""spyware""",26356
"""adware""",17978
"""banker""",11472
"""dropper""",10506
"""pua""",7862
"""riskware""",5248
"""downloader""",5090
"""sms""",3658


In [62]:
plotdf.head(10)

sha256,tlsh,first_submission_date,last_analysis_date,detection_ratio,label,file_type,family,family_confidence,behavior,file_property,packer,exploit,group,first_submission_dt,week
str,str,i64,i64,str,i64,str,str,f64,list[str],list[str],list[str],list[str],list[str],datetime[μs],i64
"""0005194678ba2866b00b0b10fc7155…","""T1F217126BF3949D2FC8770032099E…",1695845675,1702928613,"""0/76""",0,"""APK""",null,null,[],[],[],[],[],2023-09-27 20:14:35,0
"""000d37d31e99c303ec456d7949e2da…","""T165371290F396947BF4544130750A…",1695806395,1701925759,"""0/76""",0,"""APK""",null,null,[],[],[],[],[],2023-09-27 09:19:55,0
"""000ed6f3039c2f137d34676908916a…","""T17397DF05FB8E1CBAD0F3D8368A4E…",1695901528,1702364427,"""0/76""",0,"""APK""",null,null,[],[],[],[],[],2023-09-28 11:45:28,0
"""001e73a096a8afb9e537e8cba8c2c8…","""T1F3D62262E74DE81BC077A03296A2…",1695872465,1696187103,"""11/76""",1,"""APK""","""jiagu""",0.322,[],[],[],[],[],2023-09-28 03:41:05,0
"""001e97ea54db52c78241f355963308…","""T1A0C7334B9EF8D98FCD6B4036D8E8…",1695672470,1695755415,"""17/76""",1,"""APK""","""wacatac""",0.172,[],[],[],[],[],2023-09-25 20:07:50,0
"""001f34f23a1f6ea2bbde4ad41db718…","""T156773351FAA4E807E1E6D3379734…",1695850798,1695928657,"""21/75""",1,"""APK""","""jiagu""",0.181,"[""adware""]",[],[],[],[],2023-09-27 21:39:58,0
"""002d98a2ecce1440b791a00164472c…","""T11186129BFB655B3EC477C03349A9…",1695546407,1701148304,"""0/76""",0,"""APK""",null,null,[],[],[],[],[],2023-09-24 09:06:47,0
"""0030d0645e7ba2a4860bc1680f2ffd…","""T1B4E712C3C11499A9E2A34236E033…",1695613123,1702271505,"""0/76""",0,"""APK""",null,null,[],[],[],[],[],2023-09-25 03:38:43,0
"""0049ca92831a7dc7ef8f633ceb500f…","""T1C927334BAFA4FDC6F6BEA4345644…",1695913362,1702962751,"""0/78""",0,"""APK""",null,null,[],[],[],[],[],2023-09-28 15:02:42,0
